# SPAD free-form core: no assumed architecture (T4 x2)

The other SPAD runs give the learner a fixed datapath (candidate register, comparator, confidence
counter) and learn a 48-gate controller. This run assumes none of it: the circuit sees each photon's
raw 10-bit arrival time plus its own flip-flops, and every gate and every wire is learned.

Attach the private **spad-stream-design** dataset (the same one as the earlier batches), choose
**GPU T4 x2**, then **Save Version > Save & Run All**. Download `freeform_runs.zip` when it finishes.

Training aids (curriculum, scoring the answer after photons 16/32/64/128, a per-bit loss and
larger search samples) are on by default; three jobs switch them off to measure what they add.
`plain-seed0` matches the first local trial's method.

Expect this to be much harder than the controller search. Each run prints a reference line: the
error of a circuit that ignores the photons. A run only means something if it beats that and passes
the time-shift test in its summary.

In [ ]:
import glob, json, os, shutil, subprocess, sys

matches = glob.glob('/kaggle/input/**/code/spad/dataset/photons.py', recursive=True)
assert matches, 'Attach the spad-stream-design dataset'
BUNDLE = matches[0].rsplit('/code/', 1)[0]
WORK = '/kaggle/working'
CODE, DATA, RUNS = f'{WORK}/code', f'{BUNDLE}/data', f'{WORK}/freeform_runs'
SESSION_SECONDS = 8 * 3600   # stay safely below Kaggle's session limit

shutil.copytree(f'{BUNDLE}/code', CODE, dirs_exist_ok=True)
os.makedirs(f'{CODE}/freeform', exist_ok=True)
os.makedirs(RUNS, exist_ok=True)
splits = {k: v['split'] for k, v in json.load(open(f'{DATA}/processed/metadata.json'))['captures'].items()}
held_out = [k for k, v in splits.items() if v not in ('train', 'architecture', 'validation')]
assert not held_out, f'Held-out captures must not be here: {held_out}'
print(BUNDLE, len(splits), 'captures')

The model, trainer and tests, written into the working folder. These cells are copies of the files in `SPAD/freeform/`.

In [ ]:
%%writefile /kaggle/working/code/freeform/__init__.py
"""Free-form photon-stream core: no assumed datapath, every gate and flip-flop is learned."""

In [ ]:
%%writefile /kaggle/working/code/freeform/free_model.py
"""Free-form photon-stream core: nothing but learned gates and flip-flops.

The other SPAD models give the learner a fixed datapath (a candidate register, a
comparator with distance flags, a confidence counter) and learn only a small
controller. This one assumes none of that. Per photon, the circuit sees:

  the raw arrival time as `arrival_bits` binary wires, const0, const1,
  and its own `state_bits` flip-flops (all zero before the first photon).

Layers of two-input gates follow. A gate may read any input, any flip-flop, or any
gate in an earlier layer; the wiring is learned (DARTS softmax routing) and so is
each gate's 4-entry truth table. The first `state_bits` gates of the last layer
are the flip-flops' next values. After the last photon, flip-flops 0..arrival_bits-1
are read as a binary number: the range bin.

Two things are still decided by hand, because a circuit needs an interface: the
input is a binary arrival time, and the output is a binary number in named
flip-flops. Flip-flops hold their value on clocks without a photon (a clock enable).
"""
from dataclasses import asdict, dataclass

import torch
from torch import nn

FORMAT = "photon-free-core-v1"


@dataclass
class FreeConfig:
    state_bits: int = 32
    width: int = 48
    layers: int = 4
    arrival_bits: int = 10

    def __post_init__(self):
        if self.width < self.state_bits:
            raise ValueError("Last layer must hold every flip-flop's next value")
        if self.state_bits < self.arrival_bits:
            raise ValueError("Need at least arrival_bits flip-flops to hold the answer")
        if self.layers < 1:
            raise ValueError("Need at least one layer")

    @property
    def base_bits(self):
        return self.arrival_bits + 2 + self.state_bits

    def sources(self, layer):
        """How many wires a gate in `layer` may read: inputs, flip-flops, all earlier gates."""
        return self.base_bits + self.width * layer


def input_names(c):
    return ([f"arrival{i}" for i in range(c.arrival_bits)] + ["const0", "const1"]
            + [f"state{i}" for i in range(c.state_bits)])


def photon_bits(photons, bits):
    """[rays, steps] arrival bins -> ([steps, rays, bits] 0/1 floats LSB first, [steps, rays] valid)."""
    valid = photons >= 0
    value = photons.clamp(min=0).long()
    planes = (value[..., None] >> torch.arange(bits, device=photons.device)) & 1
    return planes.transpose(0, 1).float().contiguous(), valid.T.contiguous()


class FreeCore(nn.Module):
    def __init__(self, config=None, init="residual"):
        super().__init__()
        self.config = c = config or FreeConfig()
        self.routes = nn.ParameterList()
        self.table_logits = nn.ParameterList()
        for layer in range(c.layers):
            sources = c.sources(layer)
            routes = torch.randn(c.width, 2, sources) * .01
            picks = torch.randint(sources, (c.width, 2))
            routes.scatter_add_(-1, picks[..., None], torch.full((c.width, 2, 1), 2.))
            if init == "residual":
                # Generic, not task-specific: each gate starts by passing on its first
                # (randomly chosen) input, so signals and gradients cross layers and time.
                tables = torch.tensor([-2., -2., 2., 2.]).repeat(c.width, 1) + torch.randn(c.width, 4) * .5
            else:
                tables = torch.randn(c.width, 4)
            self.routes.append(nn.Parameter(routes))
            self.table_logits.append(nn.Parameter(tables))

    def architecture_parameters(self):
        return list(self.routes)

    def weight_parameters(self):
        return list(self.table_logits)

    def forward(self, photons, temperature=1., ste=False, trajectory=False):
        """Soft (or straight-through binary) pass. Returns the range as a float per ray.

        With trajectory=True, returns (values [steps, rays], output bits [steps, rays, arrival_bits]):
        the answer the circuit would give if read out after each photon.
        """
        c = self.config
        layers = []
        for routes, logits in zip(self.routes, self.table_logits):
            probabilities = (routes / temperature).softmax(-1)
            table = logits.sigmoid()
            if ste:
                chosen = torch.nn.functional.one_hot(probabilities.argmax(-1), probabilities.shape[-1]).to(table.dtype)
                probabilities = probabilities + (chosen - probabilities).detach()
                table = table + ((table > .5).to(table.dtype) - table).detach()
            width, _, sources = probabilities.shape
            weight = probabilities.transpose(0, 1).reshape(2 * width, sources).T   # all a inputs, then all b
            t00, t01, t10, t11 = table.unbind(-1)
            layers.append((weight, t00, t01 - t00, t10 - t00, t11 - t10 - t01 + t00))
        bits, valid = photon_bits(photons, c.arrival_bits)
        rays = len(photons)
        state = bits.new_zeros(rays, c.state_bits)
        consts = torch.tensor([0., 1.], device=bits.device).expand(rays, 2)
        readouts = []
        for k in range(len(bits)):
            sources = torch.cat((bits[k], consts, state), -1)
            for weight, c0, cb, ca, cab in layers:
                a, b = (sources @ weight).chunk(2, -1)
                out = c0 + a * ca + b * cb + a * b * cab
                sources = torch.cat((sources, out), -1)
            state = torch.where(valid[k][:, None], out[:, :c.state_bits], state)
            if trajectory:
                readouts.append(state[:, :c.arrival_bits])
        weights = 2. ** torch.arange(c.arrival_bits, device=bits.device)
        if trajectory:
            output = torch.stack(readouts)
            return output @ weights, output
        return state[:, :c.arrival_bits] @ weights

    @torch.no_grad()
    def circuit(self):
        layers = [{"inputs": routes.argmax(-1).tolist(), "truth_tables": (logits > 0).int().tolist()}
                  for routes, logits in zip(self.routes, self.table_logits)]
        return {"format": FORMAT, "config": asdict(self.config), "inputs": input_names(self.config),
                "output": [f"state{i}" for i in range(self.config.arrival_bits)], "layers": layers}

    @torch.no_grad()
    def load_circuit(self, circuit, sharpness=4.):
        for routes, logits, spec in zip(self.routes, self.table_logits, circuit["layers"]):
            routes.normal_(0, .01)
            routes.scatter_add_(-1, torch.tensor(spec["inputs"], device=routes.device)[..., None],
                                torch.full((*routes.shape[:2], 1), sharpness, device=routes.device))
            logits.copy_((torch.tensor(spec["truth_tables"], device=logits.device).float() * 2 - 1) * sharpness / 2)


@torch.no_grad()
def simulate_free(circuit, photons):
    """Plain integer simulation of an exported circuit, one photon at a time. The reference."""
    c = FreeConfig(**circuit["config"])
    device = photons.device
    rays = len(photons)
    state = torch.zeros(rays, c.state_bits, dtype=torch.long, device=device)
    specs = [(torch.tensor(x["inputs"], device=device), torch.tensor(x["truth_tables"], device=device))
             for x in circuit["layers"]]
    gates = torch.arange(c.width, device=device)[None]
    shifts = torch.arange(c.arrival_bits, device=device)
    consts = torch.tensor([0, 1], device=device).expand(rays, 2)
    for step in range(photons.shape[1]):
        arrival, valid = photons[:, step].long(), photons[:, step] >= 0
        pool = torch.cat(((arrival.clamp(min=0)[:, None] >> shifts) & 1, consts, state), 1)
        for inputs, tables in specs:
            output = tables[gates, 2 * pool[:, inputs[:, 0]] + pool[:, inputs[:, 1]]]
            pool = torch.cat((pool, output), 1)
        state = torch.where(valid[:, None], output[:, :c.state_bits], state)
    return (state[:, :c.arrival_bits] << shifts).sum(-1)


def pack_circuit(circuit, device=None):
    """Wiring [layers, width, 2] and truth tables as 4-bit codes [layers, width] (bit 2a+b)."""
    inputs = torch.tensor([x["inputs"] for x in circuit["layers"]], device=device)
    tables = torch.tensor([x["truth_tables"] for x in circuit["layers"]], device=device)
    return inputs, (tables << torch.arange(4, device=device)).sum(-1).int()


def unpack_circuit(template, inputs, codes):
    circuit = {key: value for key, value in template.items() if key != "layers"}
    circuit["layers"] = [{"inputs": wires.tolist(), "truth_tables": [[(code >> bit) & 1 for bit in range(4)] for code in row]}
                         for wires, row in zip(inputs.cpu(), codes.cpu().tolist())]
    return circuit


def _pack_bits(bits):
    """Booleans [..., rays] -> int64 words [..., ceil(rays/64)], 64 rays per word."""
    rays = bits.shape[-1]
    words = -(-rays // 64)
    bits = torch.nn.functional.pad(bits.long(), (0, words * 64 - rays)).reshape(*bits.shape[:-1], words, 64)
    return (bits << torch.arange(64, device=bits.device)).sum(-1)   # distinct powers: sum is OR


def _unpack_bits(words, rays):
    shifts = torch.arange(64, device=words.device)
    return ((words[..., None] >> shifts) & 1).reshape(*words.shape[:-1], -1)[..., :rays]


def pack_photons(photons, bits):
    """Pack once per photon sample: (arrival planes [steps, bits, words], valid [steps, words])."""
    value = photons.clamp(min=0).long()
    planes = ((value.T[:, None] >> torch.arange(bits, device=photons.device)[None, :, None]) & 1)
    return _pack_bits(planes), _pack_bits((photons >= 0).T)


@torch.no_grad()
def simulate_population_packed(config, inputs, codes, packed, rays, readouts=None):
    """Exact simulation of many circuits at once, 64 rays per machine word.

    inputs [members, layers, width, 2], codes [members, layers, width], packed from
    pack_photons. Returns range bins [members, rays], or with `readouts` (photon counts,
    e.g. (16, 64)) the answers read out after each of those photons [len(readouts) + 1, members,
    rays], the last row being the final answer. Each gate is evaluated in
    algebraic normal form: t00 ^ a&(t00^t10) ^ b&(t00^t01) ^ a&b&(t00^t01^t10^t11).
    """
    c = config
    planes, valid = packed
    members, words = len(inputs), planes.shape[-1]
    mask = lambda bit: -(bit.long())[..., None]                     # 0 or all ones, [members, width, 1]
    layers = []
    for layer in range(c.layers):
        code = codes[:, layer].long()
        t00, t01, t10, t11 = (code & 1, (code >> 1) & 1, (code >> 2) & 1, (code >> 3) & 1)
        index = inputs[:, layer].long()
        layers.append((index[:, :, 0, None].expand(-1, -1, words), index[:, :, 1, None].expand(-1, -1, words),
                       mask(t00), mask(t00 ^ t10), mask(t00 ^ t01), mask(t00 ^ t01 ^ t10 ^ t11)))
    state = torch.zeros(members, c.state_bits, words, dtype=torch.long, device=planes.device)
    consts = torch.stack((torch.zeros(words, dtype=torch.long, device=planes.device),
                          torch.full((words,), -1, dtype=torch.long, device=planes.device)))[None].expand(members, -1, -1)
    shifts = torch.arange(c.arrival_bits, device=planes.device)[None, :, None]
    value = lambda s: (_unpack_bits(s[:, :c.arrival_bits], rays) << shifts).sum(1)
    marks, taken = set(readouts or ()), []
    for k in range(len(planes)):
        pool = torch.cat((planes[k][None].expand(members, -1, -1), consts, state), 1)
        for ia, ib, m0, ma, mb, mab in layers:
            a, b = pool.gather(1, ia), pool.gather(1, ib)
            out = m0 ^ (a & ma) ^ (b & mb) ^ (a & b & mab)
            pool = torch.cat((pool, out), 1)
        keep = valid[k]
        state = (out[:, :c.state_bits] & keep) | (state & ~keep)
        if k + 1 in marks:
            taken.append(value(state))
    if readouts is None:
        return value(state)
    return torch.stack(taken + [value(state)])


def live_gates(circuit):
    """Gates that can influence a flip-flop the answer depends on, and the flip-flops in use."""
    c = FreeConfig(**circuit["config"])
    base = c.base_bits
    first_state = c.arrival_bits + 2
    reads = {}                                                       # wire index -> wires it reads
    for layer, spec in enumerate(circuit["layers"]):
        for gate, (a, b) in enumerate(spec["inputs"]):
            table = spec["truth_tables"][gate]
            uses_a = table[0] != table[2] or table[1] != table[3]
            uses_b = table[0] != table[1] or table[2] != table[3]
            reads[base + layer * c.width + gate] = [w for w, used in ((a, uses_a), (b, uses_b)) if used]
    last = base + (c.layers - 1) * c.width
    live, flops, stack = set(), set(), list(range(c.arrival_bits))  # start from the output flip-flops
    while stack:
        flop = stack.pop()
        if flop in flops:
            continue
        flops.add(flop)
        wires = [last + flop]
        while wires:
            w = wires.pop()
            if w >= base:
                if w not in live:
                    live.add(w)
                    wires += reads[w]
            elif w >= first_state:
                stack.append(w - first_state)
    return {"live_gates": len(live), "total_gates": c.layers * c.width, "flip_flops_used": len(flops)}

In [ ]:
%%writefile /kaggle/working/code/freeform/train_free.py
"""Train the free-form core: gradients first, then exact mutation search.

Same data, splits and randomized conditions as the other SPAD models (truth tables
on the train split, wiring on the architecture split, circuits chosen on the
validation split, held-out captures never loaded), but the circuit starts with no
datapath at all: see free_model.py.

Phase 1 (gradient): soft gates and soft wiring, annealed, then straight-through
binary. Phase 2 (hard search): copies of the best exact circuit each get one to
three random edits (flip a truth-table bit or rewire an input); the best copy on
a sample of pixels replaces the parent if it is no worse.

Three aids, each switchable, change how training runs, not what the circuit is:
  curriculum   each phase starts with many photons and no added background and
               reaches the full conditions (16-256 photons, ambient 0-0.6) partway
               through. Validation always uses the full conditions.
  readouts     the answer is also scored after photons 16, 32, 64 and 128, not only
               at the end, so partial progress is rewarded (both phases). The gradient
               phase also scores each output bit (Brier score), a denser signal than
               the single 10-bit number.
  samples      the search scores mutants on larger pixel samples and keeps a sample
               for several generations, so an accepted change is a real gain more
               often than sampling noise.

Usage (from SPAD/):
  ../.venv/bin/python -m freeform.train_free --out runs/freeform-test --steps 300 --generations 200
"""
import argparse
from dataclasses import asdict
import json
import math
from pathlib import Path
import time

import torch

from spad import ROOT
from spad.dataset.photons import load_rays, sample_conditions
from spad.training.baselines import metrics
from freeform.free_model import (FreeConfig, FreeCore, live_gates, pack_circuit, pack_photons, simulate_free,
                                 simulate_population_packed, unpack_circuit)


def centred_loss(prediction, targets, delta=4.):
    """Huber loss after removing a constant offset (a calibration constant, as in the other models)."""
    error = prediction - targets
    error = error - error.median(-1, keepdim=True).values.detach()
    return torch.nn.functional.huber_loss(error, torch.zeros_like(error), delta=delta, reduction="none").mean(-1)


class Validator:
    """Fixed validation photons under randomized conditions; offset fitted on train pixels."""
    def __init__(self, train, validation, conditions, sets, seed=7):
        device = train[0].device
        generator = torch.Generator(device=device).manual_seed(seed)
        pick = torch.randperm(len(train[1]), generator=generator, device=device)[:8000]
        self.fit_targets = train[1][pick]
        self.fit_photons = sample_conditions(train[0][pick], *conditions, generator)[0]
        draws = [sample_conditions(validation[0], *conditions, generator) for _ in range(sets)]
        self.photons, self.counts, self.background = (torch.cat(x) for x in zip(*draws))
        self.targets = validation[1].repeat(sets)

    def offset(self, circuit):
        return float((simulate_free(circuit, self.fit_photons).float() - self.fit_targets).median())

    def __call__(self, circuit):
        return metrics(simulate_free(circuit, self.photons).float() - self.offset(circuit), self.targets)

    def reference(self):
        """What a circuit that ignores the photons would score: always the median training range."""
        constant = self.fit_targets.median()
        return metrics(constant.expand(len(self.targets)), self.targets)

    def audit(self, circuit, shift=24):
        raw = simulate_free(circuit, self.photons).float()
        prediction = raw - self.offset(circuit)
        report = {"overall": metrics(prediction, self.targets)}
        for low, high in ((16, 32), (32, 64), (64, 128), (128, 257)):
            keep = (self.counts >= low) & (self.counts < high)
            if keep.any():
                row = metrics(prediction[keep], self.targets[keep])
                row["median_signed_bins"] = float((prediction[keep] - self.targets[keep]).median())
                report[f"photons_{low}-{min(high, 256)}"] = row
        # A range estimator must move with the photons. A circuit that memorized typical
        # ranges instead of measuring fails this: delay every photon and see if the answer follows.
        latest = self.photons.max(1).values
        movable = (latest + shift < 672) & (latest >= 0)
        moved = torch.where(self.photons >= 0, self.photons + shift, self.photons)[movable]
        change = simulate_free(circuit, moved).float() - raw[movable]
        report["time_shift_test"] = {"shift_bins": shift, "rays": int(movable.sum()),
                                     "median_change_bins": float(change.median()),
                                     "moves_with_photons_within_1_bin": float(((change - shift).abs() <= 1).float().mean()),
                                     "moves_with_photons_within_8_bins": float(((change - shift).abs() <= 8).float().mean())}
        return report


def readout_loss(values, targets, readouts, weight):
    """Final-answer loss plus `weight` times the mean loss of the answers after `readouts` photons.

    values: [steps or readouts + 1, ..., rays], last row final. Returns [...] losses.
    """
    final = centred_loss(values[-1], targets)
    if not weight or len(values) == 1:
        return final
    early = torch.stack([centred_loss(v, targets) for v in values[:-1]]).mean(0)
    return (final + weight * early) / (1 + weight)


def bit_loss(bits, values, targets, arrival_bits):
    """Brier score of each output bit against the target's bits, after the circuit's own offset.

    Bits are weighted by 2^(i/2): high bits matter more, low bits still count.
    """
    offset = (values[-1] - targets).median().round().detach()
    goal = (targets.round() + offset).clamp(0, 2 ** arrival_bits - 1).long()
    shifts = torch.arange(arrival_bits, device=targets.device)
    goal_bits = ((goal[:, None] >> shifts) & 1).float()
    weights = 2. ** (shifts / 2.)
    return ((bits - goal_bits).square() @ (weights / weights.sum())).mean()


def mutate(config, inputs, codes, count, generator):
    """`count` copies of a circuit, each with one to three random edits."""
    device = inputs.device
    inputs = inputs[None].repeat(count, 1, 1, 1)
    codes = codes[None].repeat(count, 1, 1)
    members = torch.arange(count, device=device)
    rand = lambda n: torch.rand(n, generator=generator, device=device)
    randint = lambda high, n: torch.randint(high, (n,), generator=generator, device=device)
    for edit, chance in enumerate((1., .4, .15)):
        chosen = members if edit == 0 else members[rand(count) < chance]
        n = len(chosen)
        layer, gate = randint(config.layers, n), randint(config.width, n)
        table = rand(n) < .5
        flipped = codes[chosen, layer, gate] ^ (1 << randint(4, n)).int()
        codes[chosen, layer, gate] = torch.where(table, flipped, codes[chosen, layer, gate])
        port = randint(2, n)
        sources = config.base_bits + config.width * layer
        pick = (rand(n) * sources).long()
        inputs[chosen, layer, gate, port] = torch.where(table, inputs[chosen, layer, gate, port], pick)
    return inputs, codes


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", type=Path, default=ROOT/"data")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--state-bits", type=int, default=32)
    parser.add_argument("--width", type=int, default=48)
    parser.add_argument("--layers", type=int, default=4)
    parser.add_argument("--init", choices=("residual", "random"), default="residual")
    parser.add_argument("--min-photons", type=int, default=16)
    parser.add_argument("--max-photons", type=int, default=256)
    parser.add_argument("--max-background", type=float, default=.6)
    parser.add_argument("--curriculum", type=float, default=.5,
                        help="fraction of each phase spent ramping from easy to full conditions (0 = off)")
    parser.add_argument("--easy-min-photons", type=int, default=128, help="fewest photons at the start of the curriculum")
    parser.add_argument("--readouts", default="16,32,64,128", help="also score the answer after these photons ('' = off)")
    parser.add_argument("--readout-weight", type=float, default=1.)
    parser.add_argument("--bit-weight", type=float, default=50.,
                        help="weight of the per-bit Brier score in the gradient phase (0 = off)")
    parser.add_argument("--resample-every", type=int, default=4, help="search generations per pixel sample")
    parser.add_argument("--steps", type=int, default=1500, help="gradient steps (0 skips phase 1)")
    parser.add_argument("--ste-fraction", type=float, default=.5, help="last part of phase 1 run straight-through")
    parser.add_argument("--batch-size", type=int, default=512)
    parser.add_argument("--lr", type=float, default=.02)
    parser.add_argument("--architecture-lr", type=float, default=.02)
    parser.add_argument("--start-temperature", type=float, default=2.)
    parser.add_argument("--end-temperature", type=float, default=.2)
    parser.add_argument("--generations", type=int, default=3000, help="hard-search generations (0 skips phase 2)")
    parser.add_argument("--population", type=int, default=512)
    parser.add_argument("--search-rays", type=int, default=4096)
    parser.add_argument("--validate-every", type=int, default=50)
    parser.add_argument("--validation-sets", type=int, default=2)
    parser.add_argument("--max-seconds", type=float, default=0, help="stop cleanly after this long (0 = no limit)")
    args = parser.parse_args()

    started = time.time()
    out_of_time = lambda: args.max_seconds and time.time() - started > args.max_seconds
    torch.manual_seed(args.seed)
    device = torch.device(args.device)
    args.out.mkdir(parents=True, exist_ok=True)
    config = FreeConfig(args.state_bits, args.width, args.layers)
    conditions = (args.min_photons, args.max_photons, args.max_background)
    readouts = tuple(int(x) for x in args.readouts.split(",") if x)

    def schedule(progress):
        """Training conditions at `progress` (0-1) through a phase: easy first, then full."""
        r = 1. if not args.curriculum else min(progress / args.curriculum, 1.)
        easy, hard = math.log(args.easy_min_photons), math.log(args.min_photons)
        return (math.exp(easy + (hard - easy) * r), args.max_photons, args.max_background * r)
    train, routing, validation = ([x.to(device) for x in load_rays(args.data, split)[:2]]
                                  for split in ("train", "architecture", "validation"))
    generator = torch.Generator(device=device).manual_seed(args.seed)
    validator = Validator(train, validation, conditions, args.validation_sets)
    reference = validator.reference()
    print(f"{device}: train {len(train[1])} rays, architecture {len(routing[1])}, validation {len(validator.targets)}; "
          f"{config.layers} x {config.width} gates, {config.state_bits} flip-flops", flush=True)
    print(f"reference (ignores photons, always the median range): MAE {reference['mae_m']:.3f} m", flush=True)

    model = FreeCore(config, args.init).to(device)
    history = []
    best = {"circuit": model.circuit(), "mae": None, "where": None}

    def consider(circuit, where):
        score = validator(circuit)
        history.append({**where, **score, "seconds": time.time() - started})
        if best["mae"] is None or score["mae_m"] < best["mae"]:
            best.update(circuit=circuit, mae=score["mae_m"], where=where)
            (args.out/"circuit.json").write_text(json.dumps(circuit) + "\n")
        return score["mae_m"]

    def batch(split, rays, progress):
        pick = torch.randint(len(split[1]), (rays,), generator=generator, device=device)
        return sample_conditions(split[0][pick], *schedule(progress), generator)[0], split[1][pick]

    consider(model.circuit(), {"phase": "start"})
    print(f"start: validation MAE {best['mae']:.3f} m", flush=True)

    # ---------------------------------------------------------------- phase 1: gradients
    weights, architecture = model.weight_parameters(), model.architecture_parameters()
    weight_optimizer = torch.optim.Adam(weights, lr=args.lr)
    architecture_optimizer = torch.optim.Adam(architecture, lr=args.architecture_lr)
    skipped = 0
    for step in range(1, args.steps + 1):
        if args.max_seconds and time.time() - started > .35 * args.max_seconds:   # leave time for the search
            print(f"gradient phase stopped at step {step} to leave time for the search", flush=True)
            break
        progress = step / args.steps
        temperature = args.start_temperature * (args.end_temperature / args.start_temperature) ** progress
        ste = progress > 1 - args.ste_fraction
        for split, optimizer, active, frozen in ((routing, architecture_optimizer, architecture, weights),
                                                 (train, weight_optimizer, weights, architecture)):
            for p in frozen:
                p.requires_grad_(False)
            photons, targets = batch(split, args.batch_size, progress)
            values, bits = model(photons, temperature, ste, trajectory=True)
            marks = [k - 1 for k in readouts if k < len(values)]
            loss = readout_loss(values[marks + [-1]], targets, readouts, args.readout_weight)
            if args.bit_weight:
                loss = loss + args.bit_weight * bit_loss(bits[marks + [-1]], values, targets, config.arrival_bits)
            optimizer.zero_grad()
            loss.backward()
            if all(torch.isfinite(p.grad).all() for p in active if p.grad is not None):
                torch.nn.utils.clip_grad_norm_(active, 1.)
                optimizer.step()
            else:
                skipped += 1
            for p in frozen:
                p.requires_grad_(True)
        if step % 50 == 0 or step == args.steps:
            mae = consider(model.circuit(), {"phase": "gradient", "step": step, "temperature": temperature,
                                             "straight_through": ste, "loss": float(loss.detach())})
            print(f"step {step:5d}  T {temperature:.2f}  {'ste ' if ste else 'soft'}  loss {float(loss.detach()):8.2f}  "
                  f"hard validation MAE {mae:.3f} m (best {best['mae']:.3f})  {time.time() - started:.0f} s", flush=True)
    gradient_best = best["mae"]

    # ---------------------------------------------------------------- phase 2: exact search
    inputs, codes = pack_circuit(best["circuit"], device)
    template = best["circuit"]
    accepted = improved = generation = 0
    for generation in range(1, args.generations + 1):
        if out_of_time():
            break
        if (generation - 1) % args.resample_every == 0:
            sample = (generation - 1) // args.resample_every
            photons, targets = batch(train if sample % 2 == 0 else routing, args.search_rays,
                                     generation / args.generations)
            packed = pack_photons(photons, config.arrival_bits)
        new_inputs, new_codes = mutate(config, inputs, codes, args.population, generator)
        all_inputs, all_codes = torch.cat((inputs[None], new_inputs)), torch.cat((codes[None], new_codes))
        prediction = simulate_population_packed(config, all_inputs, all_codes, packed, len(photons),
                                                readouts=readouts or None).float()
        if not readouts:
            prediction = prediction[None]
        scores = readout_loss(prediction, targets[None], readouts, args.readout_weight)
        winner = int(scores[1:].argmin()) + 1
        if scores[winner] <= scores[0]:
            accepted += 1
            improved += int(scores[winner] < scores[0])
            inputs, codes = all_inputs[winner], all_codes[winner]
        if generation % args.validate_every == 0 or generation == args.generations:
            mae = consider(unpack_circuit(template, inputs, codes),
                           {"phase": "hard", "generation": generation, "sample_loss": float(scores[0])})
            print(f"generation {generation:5d}  sample loss {float(scores[0]):8.2f}  validation MAE {mae:.3f} m "
                  f"(best {best['mae']:.3f})  accepted {accepted} improved {improved}  {time.time() - started:.0f} s",
                  flush=True)

    circuit = best["circuit"]
    summary = {"format": circuit["format"], "config": asdict(config), "arguments": {k: str(v) for k, v in vars(args).items()},
               "validation": validator(circuit), "chosen_at": best["where"], "offset_bins": validator.offset(circuit),
               "reference_ignoring_photons": reference, "gradient_best_validation_mae_m": gradient_best,
               "generations_run": generation, "accepted": accepted, "improved": improved,
               "skipped_nonfinite_steps": skipped, "gates": live_gates(circuit), "audit": validator.audit(circuit),
               "seconds": time.time() - started}
    (args.out/"summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    (args.out/"history.json").write_text(json.dumps(history) + "\n")
    shift = summary["audit"]["time_shift_test"]
    print(json.dumps({"validation": summary["validation"], "reference_mae_m": reference["mae_m"],
                      "gates": summary["gates"], "time_shift_test": shift}, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/freeform/test_free.py
"""The three implementations of the free-form core must agree exactly.

Run from SPAD/:  ../.venv/bin/python -m unittest freeform.test_free -v
"""
import unittest

import torch

from freeform.free_model import (FreeConfig, FreeCore, live_gates, pack_circuit, pack_photons, simulate_free,
                                 simulate_population_packed, unpack_circuit)


def random_model(seed, config):
    torch.manual_seed(seed)
    model = FreeCore(config, init="random")
    for routes, logits in zip(model.routes, model.table_logits):
        routes.data.normal_()
        logits.data.normal_()
    return model


class FreeCoreParity(unittest.TestCase):
    def setUp(self):
        self.config = FreeConfig(state_bits=12, width=20, layers=3)
        torch.manual_seed(0)
        self.photons = torch.randint(-1, 672, (200, 40))

    def test_straight_through_forward_equals_integer_circuit(self):
        for seed in range(3):
            model = random_model(seed, self.config)
            expected = simulate_free(model.circuit(), self.photons)
            got = model(self.photons, temperature=.5, ste=True)
            self.assertTrue(torch.equal(got.round().long(), expected))

    def test_packed_population_equals_integer_circuit(self):
        circuits = [random_model(seed, self.config).circuit() for seed in range(4)]
        packed = [pack_circuit(c) for c in circuits]
        inputs, codes = torch.stack([p[0] for p in packed]), torch.stack([p[1] for p in packed])
        got = simulate_population_packed(self.config, inputs, codes,
                                         pack_photons(self.photons, self.config.arrival_bits), len(self.photons))
        for k, circuit in enumerate(circuits):
            self.assertTrue(torch.equal(got[k], simulate_free(circuit, self.photons)))

    def test_readouts_after_each_photon(self):
        model = random_model(3, self.config)
        circuit = model.circuit()
        values, bits = model(self.photons, temperature=.5, ste=True, trajectory=True)
        self.assertEqual(tuple(bits.shape), (40, 200, self.config.arrival_bits))
        inputs, codes = pack_circuit(circuit)
        packed = simulate_population_packed(self.config, inputs[None], codes[None],
                                            pack_photons(self.photons, self.config.arrival_bits), 200, readouts=(5, 17))
        for row, k in enumerate((5, 17, 40)):
            expected = simulate_free(circuit, self.photons[:, :k])
            self.assertTrue(torch.equal(values[k - 1].round().long(), expected))
            self.assertTrue(torch.equal(packed[row, 0], expected))

    def test_pack_round_trip_and_load(self):
        model = random_model(5, self.config)
        circuit = model.circuit()
        self.assertEqual(unpack_circuit(circuit, *pack_circuit(circuit)), circuit)
        other = FreeCore(self.config)
        other.load_circuit(circuit)
        self.assertEqual(other.circuit(), circuit)

    def test_flip_flops_hold_without_a_photon(self):
        circuit = random_model(1, self.config).circuit()
        dense = torch.randint(0, 672, (50, 10))
        gaps = torch.full((50, 20), -1)
        gaps[:, ::2] = dense
        self.assertTrue(torch.equal(simulate_free(circuit, gaps), simulate_free(circuit, dense)))

    def test_live_gate_count_is_bounded(self):
        report = live_gates(random_model(2, self.config).circuit())
        self.assertLessEqual(report["live_gates"], report["total_gates"])
        self.assertGreaterEqual(report["flip_flops_used"], self.config.arrival_bits)


if __name__ == "__main__":
    unittest.main()

The three implementations (training pass, integer circuit, bit-packed search) must agree exactly on this GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose GPU T4 x2 under Accelerator'
GPUS = torch.cuda.device_count()
print(torch.__version__, GPUS, [torch.cuda.get_device_name(i) for i in range(GPUS)])
subprocess.run([sys.executable, '-m', 'unittest', 'freeform.test_free', '-v'], cwd=CODE, check=True)

In [ ]:
# name, then train_free.py arguments. Two jobs run at once, one per GPU.
JOBS = [
    ('aids-seed0',          '--seed 0'),                      # curriculum + readouts + bit loss + larger samples
    ('aids-seed1',          '--seed 1'),
    ('aids-wide-seed0',     '--seed 0 --layers 6 --width 64 --state-bits 48'),
    ('no-curriculum-seed0', '--seed 0 --curriculum 0'),        # each aid switched off once, to see what it adds
    ('final-only-seed0',    '--seed 0 --readouts "" --bit-weight 0'),
    ('plain-seed0',         '--seed 0 --curriculum 0 --readouts "" --bit-weight 0 --resample-every 1 --search-rays 2048'),
]
STEPS, BATCH, GENERATIONS, POPULATION, SEARCH_RAYS = 1500, 512, 4000, 512, 4096
PER_JOB_SECONDS = int(SESSION_SECONDS * GPUS / len(JOBS) * .9)   # every job gets a fair share of the session

import queue, shlex, threading
todo = queue.Queue()
for job in JOBS:
    todo.put(job)

def worker(gpu):
    while True:
        try:
            name, flags = todo.get_nowait()
        except queue.Empty:
            return
        if os.path.exists(f'{RUNS}/{name}/summary.json'):
            continue
        os.makedirs(f'{RUNS}/{name}', exist_ok=True)
        command = [sys.executable, '-m', 'freeform.train_free', '--data', DATA, '--out', f'{RUNS}/{name}',
                   '--steps', str(STEPS), '--batch-size', str(BATCH), '--generations', str(GENERATIONS),
                   '--population', str(POPULATION), '--search-rays', str(SEARCH_RAYS),
                   '--max-seconds', str(PER_JOB_SECONDS), *shlex.split(flags)]
        with open(f'{RUNS}/{name}/log.txt', 'w') as log:
            done = subprocess.run(command, cwd=CODE, stdout=log, stderr=subprocess.STDOUT,
                                  env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)})
        tail = open(f'{RUNS}/{name}/log.txt').read().splitlines()[-3:]
        print(f'[gpu {gpu}] {name} exit {done.returncode}:', *tail, sep='\n   ', flush=True)

threads = [threading.Thread(target=worker, args=(gpu,)) for gpu in range(GPUS)]
[t.start() for t in threads]
[t.join() for t in threads]

In [ ]:
rows = []
for path in glob.glob(f'{RUNS}/*/summary.json'):
    s = json.load(open(path))
    shift = s['audit']['time_shift_test']
    rows.append((s['validation']['mae_m'], os.path.basename(os.path.dirname(path)), s['validation']['median_m'],
                 s['reference_ignoring_photons']['mae_m'], shift['moves_with_photons_within_8_bins'],
                 s['gates']['live_gates'], s['gates']['flip_flops_used'], s['seconds']))
print(f"{'run':20s} {'MAE m':>7s} {'median':>7s} {'ignore-photons MAE':>19s} {'follows a time shift':>21s} {'gates':>6s} {'flops':>6s} {'min':>6s}")
for mae, name, median, ref, follows, gates, flops, seconds in sorted(rows):
    print(f'{name:20s} {mae:7.3f} {median:7.3f} {ref:19.3f} {follows:21.1%} {gates:6d} {flops:6d} {seconds/60:6.1f}')
print('For comparison, the fixed-datapath final design (10 Nov): MAE 0.363 m, median 0.151 m.')
shutil.make_archive(f'{WORK}/freeform_runs', 'zip', RUNS)